## Gymnasium environments

This Section shows how you can make and use the `gym` environments that interface with the simulator.

In [25]:
import os
from pathlib import Path
import torch
import mediapy

# Set working directory to the base directory 'gpudrive'
working_dir = Path.cwd()
while working_dir.name != 'gpudrive':
    working_dir = working_dir.parent
    if working_dir == Path.home():
        raise FileNotFoundError("Base directory 'gpudrive' not found")
os.chdir(working_dir)

from pygpudrive.env.config import EnvConfig, RenderConfig, SceneConfig, SelectionDiscipline
from pygpudrive.env.env_torch import GPUDriveTorchEnv

### Settings

In [28]:
MAX_NUM_OBJECTS = 32  # Maximum number of objects in the scene we control
NUM_WORLDS = 4  # Number of parallel environments
K_UNIQUE_SCENES = 3 # Number of unique scenes

# Set the path to where you want to save the videos
VIDEO_PATH = "./videos"

FPS = 4  # Video frames per second

device = 'cuda' # for explanation purposes in notebook #torch.device("cuda" if torch.cuda.is_available() else "cpu")

### Initializing environments

- We provide both a torch and jax gymnasium interface with the simulator. Most functionality is specified in the `GPUDriveGymEnv` class in the `base_env`, `torch_env` and `jax_env` both inherit from the `GPUDriveGymEnv`, the only difference between these is that one exports torch tensors and the other jax arrays.
- All data settings are defined in the `SceneConfig` dataclass.
- All environment settings are defined in the `EnvConfig` dataclass. 
- All rendering configs are defined in the `RenderConfig` dataclass.


In [29]:
scene_config = SceneConfig(
    path="data/processed/examples", 
    num_scenes=NUM_WORLDS,
    discipline=SelectionDiscipline.K_UNIQUE_N,
    k_unique_scenes=K_UNIQUE_SCENES,
)

In [30]:
accel_discretization = 21 
accel_lower_bound = -4.0 # decelerate
accel_upper_bound = 4.0 # accelerate
steering_lower_bound = -0.3 # steer right
steering_upper_bound = 0.3 # steer left
steering_discretization = 31

env_config = EnvConfig(
    steer_actions = torch.round(
        torch.linspace(steering_lower_bound, steering_upper_bound, steering_discretization), decimals=3),
    accel_actions = torch.round(
        torch.linspace(accel_lower_bound, accel_upper_bound, accel_discretization), decimals=3),
)

'''
discretize_actions: true
include_head_angle: false # Whether to include the head tilt/angle as part of a vehicle's action
accel_discretization: 21 
accel_lower_bound: -4.0 # decelerate
accel_upper_bound: 4.0 # accelerate
steering_lower_bound: -0.3 # steer right
steering_upper_bound: 0.3 # steer left
steering_discretization: 31
'''

"\ndiscretize_actions: true\ninclude_head_angle: false # Whether to include the head tilt/angle as part of a vehicle's action\naccel_discretization: 21 \naccel_lower_bound: -4.0 # decelerate\naccel_upper_bound: 4.0 # accelerate\nsteering_lower_bound: -0.3 # steer right\nsteering_upper_bound: 0.3 # steer left\nsteering_discretization: 31\n"

In [31]:
render_config = RenderConfig(
    resolution=(800, 800), # Quality of the rendered images
    draw_obj_idx=True, # Draw object index on the objects
)

---

> 💡 **For more info about the environment configurations, see `pygpudrive/env/README.md`**

---

In [32]:
# MAKE ENV
env = GPUDriveTorchEnv(
    config=env_config,
    scene_config=scene_config,
    render_config=render_config,
    max_cont_agents=MAX_NUM_OBJECTS, # Maximum number of agents to control per scenario
    device=device,
)




--- Ratio unique scenes / number of worls = 3 / 4 ---

Compiling GPU engine code:
/gpudrive/gpudrive/external/madrona/src/mw/device/memory.cpp
/gpudrive/gpudrive/external/madrona/src/mw/device/state.cpp
/gpudrive/gpudrive/external/madrona/src/mw/device/crash.cpp
/gpudrive/gpudrive/external/madrona/src/mw/device/consts.cpp
/gpudrive/gpudrive/external/madrona/src/mw/device/taskgraph.cpp
/gpudrive/gpudrive/external/madrona/src/mw/device/taskgraph_utils.cpp
/gpudrive/gpudrive/external/madrona/src/mw/device/sort_archetype.cpp
/gpudrive/gpudrive/external/madrona/src/mw/device/host_print.cpp
/gpudrive/gpudrive/external/madrona/src/mw/../common/hashmap.cpp
/gpudrive/gpudrive/external/madrona/src/mw/../common/navmesh.cpp
/gpudrive/gpudrive/external/madrona/src/mw/../core/base.cpp
/gpudrive/gpudrive/external/madrona/src/mw/../physics/physics.cpp
/gpudrive/gpudrive/external/madrona/src/mw/../physics/geo.cpp
/gpudrive/gpudrive/external/madrona/src/mw/../physics/xpbd.cpp
/gpudrive/gpudrive/externa

/gpudrive/gpudrive/src/level_gen.cpp(280): warning #177-D: function "gpudrive::createFloorPlane" was declared but never referenced
  static void createFloorPlane(Engine &ctx)
              ^

Remark: The warnings can be suppressed with "-diag-suppress <warning-number>"





### Run an episode with `N` parallel environments

A single rollout (one episode) is implemented as follows:
- We step through N worlds simultaneously.
- Use the `world_render_idx` argument in `render(.)` to select which world to render.

In [34]:
from torch import nn

class Net(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(Net, self).__init__()
        self.nn = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.Tanh(),
            nn.Linear(hidden_size, hidden_size),
            nn.Tanh(),
        )
        self.heads = nn.ModuleList([nn.Linear(hidden_size, output_size)])
    
    def dist(self, obs):
        """Generate action distribution."""
        x_out = self.nn(obs.float())
        return [Categorical(logits=head(x_out)) for head in self.heads]
        
    def forward(self, obs, deterministic=False):
        """Generate an output from tensor input."""
        action_dist = self.dist(obs)
        
        if deterministic:
            actions_idx = action_dist[0].logits.argmax(axis=-1) 
        else:
            actions_idx = action_dist.sample()
        return actions_idx
    
    def _log_prob(self, obs, expert_actions):
        pred_action_dist = self.dist(obs)
        log_prob = pred_action_dist[0].log_prob(expert_actions).mean()
        return log_prob


model = torch.load("../nocturne_lab/models_trained/il/trained_bc_policy.pt")
model

/tmp/ipykernel_133221/3475231744.py:35: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model = torch.load("../nocturne_lab/models_trained/il/trained_bc_policy.pt")


Net(
  (nn): Sequential(
    (0): Linear(in_features=6730, out_features=800, bias=True)
    (1): Tanh()
    (2): Linear(in_features=800, out_features=800, bias=True)
    (3): Tanh()
  )
  (heads): ModuleList(
    (0): Linear(in_features=800, out_features=651, bias=True)
  )
)

In [36]:
obs = env.reset()
obs.shape

torch.Size([4, 32, 2916])

In [1]:
obs = env.reset()
frames = {f"env_{i}": [] for i in range(NUM_WORLDS)}

print(env.action_space.n)

for _ in range(env_config.episode_len):
    # SELECT ACTIONS
    rand_action = torch.Tensor(
        [[env.action_space.sample() for _ in range(MAX_NUM_OBJECTS * NUM_WORLDS)]]
    ).reshape(NUM_WORLDS, MAX_NUM_OBJECTS)
    
    print(rand_action.shape)

    # STEP
    env.step_dynamics(rand_action)

    obs = env.get_obs()
    reward = env.get_rewards()
    done = env.get_dones()

    print(obs.shape)

    # RENDER
    for i in range(NUM_WORLDS):
        frames[f"env_{i}"].append(env.render(i))

NameError: name 'env' is not defined